# 08 Problem 2: retraining with augmentation
T4 GPU, ~30-40 min. Needs nb 06 and 07 first.

Champion = ResNet-50 + aug, challenger = CLIP probe + aug, both still trained on SD1.4 only. Tested on 8 generators + SDXL-Turbo, clean and degraded.

Refs: Wang et al. 2020 (arXiv 1912.11035), UnivFD (2302.10174), ResNet (1512.03385), CLIP (2103.00020)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

## 1. GPU check

In [ ]:
import torch

ALLOW_CPU = False   # CPU is way too slow, debug only
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    msg = ("NO GPU FOUND. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart "
           "and run all again.")
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from torch.utils.data import DataLoader

from src import datasets as ds
from src import degrade as dg
from src import eda_utils as eu
from src import evaluate as ev
from src import models as mdl
from src import robustness as rb
from src import stats as st
from src import train as tr
from src.config import generator_names, get_logger
from src.manifest import read_csv

eu.set_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("08_problem2_augmentation", P.logs)
SEED = cfg["project"]["seed"]
TRAIN_GEN = cfg["train_generator"]
THR = cfg["evaluation"]["threshold"]
FEAT = cfg["features"]
RCFG = cfg["training"]["resnet50"]
P2 = cfg["problem2"]
CONDS, AUG, SX = P2["conditions"], P2["augment"], P2["sdxl"]
NAMES = {**generator_names(cfg), SX["folder"]: SX["name"]}
GEN_ORDER = [g["folder"] for g in cfg["generators"]] + [SX["folder"]]
RES, FIG = P.results / "problem2", P.figures / "problem2"
PRED_DIR, FEAT_ROOT, CK2 = RES / "preds", P.features / "problem2", P.checkpoints / "problem2"
CK2.mkdir(parents=True, exist_ok=True)

RESNET, CLIP = "ResNet-50 (fine-tuned)", "CLIP ViT-L/14 + LR"            # Problem 1 baselines
RESNET_AUG, CLIP_AUG = "ResNet-50 + aug", "CLIP ViT-L/14 + LR + aug"     # Problem 2 models
MODEL_ORDER = [RESNET, RESNET_AUG, CLIP, CLIP_AUG]
STYLES = {RESNET: {"color": "C0", "ls": "--"}, RESNET_AUG: {"color": "C0"},
          CLIP: {"color": "C1", "ls": "--"}, CLIP_AUG: {"color": "C1"}}


def save(fig, name):
    eu.savefig(fig, f"08_{name}", FIG)
    plt.show()


index = ds.load_index(P.manifests).sort_values("sample_id").reset_index(drop=True)
sdxl = read_csv(P.project_root / "problem2" / SX["folder"] / "sdxl_index.csv").sort_values("sample_id").reset_index(drop=True)
masks = {s: (index["split_set"] == s).to_numpy() for s in ["train", "val", "test"]}
TEST_SETS = {"genimage": index.loc[masks["test"]].reset_index(drop=True), "sdxl": sdxl}
print({k: len(v) for k, v in TEST_SETS.items()})

### Copy images to local disk

In [ ]:
n = ds.stage_locally(pd.concat([index["processed_relpath"], sdxl["processed_relpath"]]), P.project_root, P.work_dir)
ROOT = P.work_dir
print(f"Copied {n} new files")

## 2. Training augmentation
Random blur / resize / JPEG, each with p = 0.5, on real and fake.

In [ ]:
row = index.loc[masks["train"] & (index["y"] == 1)].iloc[0]
img = Image.open(ROOT / row["processed_relpath"]).convert("RGB")
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(img.crop((64, 64, 160, 160)))
axes[0].set_title("original", fontsize=8)
for k, ax in enumerate(axes[1:]):
    aug = dg.RandomDegrade(**AUG, deterministic=True, seed=SEED + k)
    steps = aug.sample_steps(__import__("random").Random(dg.stable_seed(SEED + k, row["sample_id"])))
    ax.imshow(aug(img, row["sample_id"]).crop((64, 64, 160, 160)))
    ax.set_title(", ".join(f"{s}{v:.2g}" for s, v in steps) or "none", fontsize=7)
for ax in axes:
    ax.axis("off")
fig.suptitle("Random training augmentation (96 px crops)", fontweight="bold")
fig.tight_layout()
save(fig, "augmentation_examples")

## 3. Champion: ResNet-50 + augmentation
Same settings as Problem 1, model picked on clean SD1.4 val, resumes from `last.pt`.

In [ ]:
g = torch.Generator().manual_seed(SEED)
train_ds = dg.DegradedDataset(index.loc[masks["train"]], ROOT, ds.resnet_transforms(RCFG["crop_size"], train=True),
                              dg.RandomDegrade(**AUG))
val_ds = ds.GenImageDataset(index.loc[masks["val"]], ROOT, ds.resnet_transforms(RCFG["crop_size"], train=False))
dl_kw = dict(batch_size=RCFG["batch_size"], num_workers=RCFG["num_workers"], pin_memory=DEVICE == "cuda")
train_dl = DataLoader(train_ds, shuffle=True, drop_last=True, generator=g, **dl_kw)
val_dl = DataLoader(val_ds, shuffle=False, **dl_kw)

CKPT = CK2 / "resnet50_aug"
resnet_aug = mdl.build_resnet50(RCFG["weights"])
history = tr.train_resnet(resnet_aug, train_dl, val_dl, val_ds.df["y"].to_numpy(), CKPT, epochs=RCFG["epochs"],
                          lr=RCFG["lr"], weight_decay=RCFG["weight_decay"], patience=RCFG["early_stopping_patience"],
                          amp=RCFG["amp"], device=DEVICE, logger=log)
display(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train (augmented)")
axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="val (clean)")
axes[0].set(title="ResNet-50 + aug loss", xlabel="epoch", ylabel="BCE")
axes[0].legend()
axes[1].plot(history["epoch"], history["val_auroc"], marker="o", label="val AUROC")
axes[1].plot(history["epoch"], history["val_acc"], marker="o", label="val accuracy")
axes[1].set(title="ResNet-50 + aug, SD1.4 val", xlabel="epoch", ylim=(0.4, 1.01))
axes[1].legend()
save(fig, "resnet50_aug_training_curves")

In [ ]:
def load_resnet(path):
    m = mdl.build_resnet50(weights=None)
    m.load_state_dict(torch.load(path, map_location="cpu", weights_only=False)["model"])
    return m.to(DEVICE).eval()


resnet_aug = load_resnet(CKPT / "best.pt")
resnet_base = load_resnet(P.checkpoints / "resnet50" / "best.pt")

## 4. Challenger: CLIP + LR on clean + augmented features
Clean features from the nb 03 cache, 2 augmented copies (fixed per image).

In [ ]:
feat_kw = dict(image_size=cfg["preprocess"]["image_size"], batch_size=FEAT["batch_size"],
               shard_size=FEAT["shard_size"], num_workers=FEAT["num_workers"], device=DEVICE)
clip = rb.LazyExtractor("clip", cfg)
X_all = rb.features_for(clip, index, ROOT, [], P.features / "clip", **feat_kw)  # nb 03 cache
y_all = index["y"].to_numpy()
tr_df = index.loc[masks["train"]].reset_index(drop=True)
X_parts, y_parts = [X_all[masks["train"]]], [y_all[masks["train"]]]
for c in range(P2["probe_aug_copies"]):
    aug = dg.RandomDegrade(**AUG, deterministic=True, seed=SEED + 1000 + c)
    X_parts.append(rb.features_for(clip, tr_df, ROOT, None, FEAT_ROOT / "clip" / f"train_aug{c}", degrade=aug, **feat_kw))
    y_parts.append(tr_df["y"].to_numpy())
X_tr, y_tr = np.concatenate(X_parts), np.concatenate(y_parts)
print(f"probe training set: {X_tr.shape} (clean + {P2['probe_aug_copies']} augmented copies)")

clip_aug, tuning = tr.train_linear_probe(X_tr, y_tr, X_all[masks["val"]], y_all[masks["val"]],
                                         cfg["training"]["linear_probe"]["C_grid"],
                                         cfg["training"]["linear_probe"]["max_iter"], SEED, logger=log)
joblib.dump(clip_aug, CK2 / "linear_probe_clip_aug.joblib")
ev.save_table(tuning.assign(model=CLIP_AUG), RES / "linear_probe_clip_aug_tuning.csv")
print(f"best C = {tuning.loc[tuning['val_auroc'].idxmax(), 'C']}")
clip_base = joblib.load(P.checkpoints / "linear_probe_clip.joblib")

## 5. Score all 4 models
CLIP features per condition are shared by both probes, baseline scores come from the nb 06 cache.

In [ ]:
# "can only test a child process" on py3.13 is harmless
parts = []
for set_name, df in TEST_SETS.items():
    for m, name in [(resnet_base, RESNET), (resnet_aug, RESNET_AUG)]:
        parts.append(rb.score_resnet(m, df, ROOT, CONDS, name, set_name, PRED_DIR, RCFG["crop_size"],
                                     RCFG["batch_size"], RCFG["num_workers"], DEVICE, RCFG["amp"], log))
    for clf, name in [(clip_base, CLIP), (clip_aug, CLIP_AUG)]:
        parts.append(rb.score_probe(clip, clf, df, ROOT, CONDS, name, set_name, PRED_DIR, FEAT_ROOT, log, **feat_kw))
clip.free()
preds = pd.concat(parts, ignore_index=True)
preds.to_csv(RES / "augmentation_predictions.csv", index=False)
res = rb.results_by_condition(preds, CONDS, GEN_ORDER, TRAIN_GEN, THR)
res.insert(3, "generator_name", res["generator"].map(NAMES).fillna(res["generator"]))
ev.save_table(res, RES / "augmentation_results.csv")

## 6. Clean test sets
`mean_unseen` = 7 GenImage generators + SDXL-Turbo.

In [ ]:
clean = res.loc[res["condition"] == "clean"]
cols = [NAMES[g] for g in GEN_ORDER] + ["mean_unseen"]
for metric in ["auroc", "accuracy", "f1", "real_acc", "fake_acc"]:
    piv = clean.pivot(index="model", columns="generator_name", values=metric)[cols].reindex(MODEL_ORDER)
    print(f"\n{metric.upper()}, clean")
    display(piv.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0, axis=None))
    piv.to_csv(RES / f"augmentation_clean_{metric}.csv")

In [ ]:
clean_preds = preds.loc[preds["condition"] == "clean"]
fig = ev.plot_metric_bars(clean.loc[clean["generator"].isin(GEN_ORDER)], "auroc", GEN_ORDER, NAMES)
save(fig, "clean_auroc_per_generator")
fig = ev.plot_roc_per_generator(clean_preds.loc[clean_preds["model"].isin([RESNET_AUG, CLIP_AUG])], GEN_ORDER, ncols=5, names=NAMES)
save(fig, "clean_roc_aug_models")
for m in [RESNET_AUG, CLIP_AUG]:
    fig = ev.plot_confusion_grid(clean_preds, m, GEN_ORDER, THR, ncols=5, names=NAMES)
    save(fig, f"confusion_{rb.slug(m)}")

## 7. Degraded test sets
Dashed = Problem 1 baseline, solid = with augmentation.

In [ ]:
for g, title in [("mean_all", "Mean AUROC, 8 GenImage generators + SDXL-Turbo"), (TRAIN_GEN, "AUROC on SD1.4 (seen)"),
                 (SX["folder"], "AUROC on SDXL-Turbo (new, unseen)")]:
    fig = rb.plot_degradation_curves(res, g, "auroc", MODEL_ORDER, STYLES, title=title)
    save(fig, f"auroc_vs_degradation_{rb.slug(NAMES.get(g, g))}")

In [ ]:
cond_order = list(CONDS)
for g in ["mean_all", "mean_unseen", SX["folder"]]:
    piv = (res.loc[res["generator"] == g].pivot(index="model", columns="condition", values="auroc")
              .reindex(index=MODEL_ORDER, columns=cond_order))
    print(f"\nAUROC, {NAMES.get(g, g)}")
    display(piv.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0, axis=None))
    gain = pd.DataFrame({"ResNet: aug - base": piv.loc[RESNET_AUG] - piv.loc[RESNET],
                         "CLIP: aug - base": piv.loc[CLIP_AUG] - piv.loc[CLIP]}).T
    print(f"Gain from augmentation, {NAMES.get(g, g)}")
    display(gain.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-0.3, vmax=0.3, axis=None))
    piv.to_csv(RES / f"augmentation_auroc_{rb.slug(NAMES.get(g, g))}.csv")

## 8. Champion vs challenger
Bootstrap CIs + paired tests on clean and social.

In [ ]:
BS = cfg["bootstrap"]
ci_rows, diff_rows = [], []
for cond in ["clean", "social"]:
    p = preds.loc[preds["condition"] == cond]
    ci_rows.append(st.bootstrap_table(p, GEN_ORDER, TRAIN_GEN, BS["n_boot"], SEED, BS["alpha"]).assign(condition=cond))
    for a, b in [(CLIP_AUG, RESNET_AUG), (RESNET_AUG, RESNET), (CLIP_AUG, CLIP)]:
        diff_rows.append(st.paired_diff_table(p, a, b, GEN_ORDER, TRAIN_GEN, BS["n_boot"], SEED, BS["alpha"]).assign(condition=cond))
ci = pd.concat(ci_rows, ignore_index=True)
diffs = pd.concat(diff_rows, ignore_index=True)
diffs["generator_name"] = diffs["generator"].map(NAMES).fillna(diffs["generator"])
ev.save_table(ci, RES / "augmentation_bootstrap_ci.csv")
ev.save_table(diffs, RES / "augmentation_paired_diffs.csv")
show = diffs.loc[diffs["generator"].isin([TRAIN_GEN, SX["folder"], "mean_unseen"])]
display(show[["condition", "model_a", "model_b", "generator_name", "diff", "lo", "hi", "significant"]].round(3))
for cond in ["clean", "social"]:
    fig = st.plot_ci(ci.loc[(ci["condition"] == cond) & ci["model"].isin([RESNET_AUG, CLIP_AUG])], GEN_ORDER, NAMES,
                     [RESNET_AUG, CLIP_AUG], title=f"Champion vs challenger, {cond}: AUROC with 95% CI")
    save(fig, f"champion_vs_challenger_ci_{cond}")

## 9. Pros and cons
- ResNet-50 + aug: trains all 25M weights (~5 epochs on a T4), small and fast to serve, has Grad-CAM. But it learns low-level traces that break on new generators.
- CLIP + LR + aug: only a logistic regression on frozen 768-d features, cheap to retrain. Generalizes better (Problem 1), but ViT-L/14 is ~300M params (needs a GPU) and augmentation can only move the linear boundary.

## Findings
Fill in after the run.
- ResNet + aug training (best epoch, val AUROC):
- Clean (SD1.4 cost, unseen, SDXL-Turbo):
- Degraded (gain from augmentation):
- Champion vs challenger (section 8):

## Run before closing the tab
Uploads pending Drive writes, otherwise the next session may not see them.

In [ ]:
from src.colab_setup import flush_drive
flush_drive()